In [3]:
#---PRODUCT DEMAND---

import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Load data
df = pd.read_csv('Historical Product Demand.csv')

#---STEP 2---

#Check for missing values 

print("Missing values per column before cleaning:")
print(df.isna().sum())

#Drop all missing values
df.dropna(inplace=True)

# 4. Verify they are gone
print("\nMissing values after cleaning:")
print(df.isna().sum())

# Clean the target 'Order_Demand' (remove text/parentheses, make it numeric)
df['Order_Demand'] = df['Order_Demand'].astype(str).str.replace('(', '', regex=False)
df['Order_Demand'] = df['Order_Demand'].str.replace(')', '', regex=False)
df['Order_Demand'] = pd.to_numeric(df['Order_Demand'], errors='coerce')

# Drop any new NaNs that resulted from conversion errors
df.dropna(subset=['Order_Demand'], inplace=True)

# Feature cleanup on Date
df['Date'] = pd.to_datetime(df['Date'])
df['Month'] = df['Date'].dt.month
df['Year'] = df['Date'].dt.year

# Define our categorical and numerical columns
cat_features = ['Warehouse', 'Product_Category']
num_features = ['Month', 'Year']

# Preprocessor pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_features)
    ],
    remainder='drop'
)

# Extract features (X) and target (y)
X = df[num_features + cat_features]
y = df['Order_Demand'].values

# Fit and transform the features
X_processed = preprocessor.fit_transform(X)

print(f"Processed feature shape: {X_processed.shape}")

#---STEP 5---
total_rows = X_processed.shape[0]

# Calculate the exact row cut-off points
train_end = int(total_rows * 0.80)
val_end = int(total_rows * 0.90)

# Slice the features (X) chronologically
X_train = X_processed[:train_end]
X_val = X_processed[train_end:val_end]
X_test = X_processed[val_end:]

# Slice the target (y) chronologically
y_train = y[:train_end]
y_val = y[train_end:val_end]
y_test = y[val_end:]

# Verify the final proportions
print(f"Train set size:      {X_train.shape[0]} rows (80%)")
print(f"Validation set size: {X_val.shape[0]} rows (10%)")
print(f"Test set size:        {X_test.shape[0]} rows (10%)")

#---STEP 6---

#  DECISION TREE REGRESSOR

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.metrics import r2_score

# 1. Mock Data Generation (Replace this with your actual data)
np.random.seed(42)
X = np.random.rand(200, 5)
y = 3 * X[:, 0] + 2 * np.exp(X[:, 1]) + np.random.randn(200)

# 2. Correct 3-Way Split (Train, Validation, Test)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42) 

# 3. Scaling (CRITICAL for SVR performance)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# 4. Hyperparameter Tuning for Decision Tree
criteria_options = ['squared_error', 'friedman_mse', 'absolute_error', 'poisson']
best_criterion = None
best_val_r2 = -float('inf')

for criterion in criteria_options:
    # Ensure target is positive if using poisson
    if criterion == 'poisson' and np.any(y_train <= 0):
        continue
        
    dt = DecisionTreeRegressor(criterion=criterion, random_state=42)
    dt.fit(X_train, y_train) # Trees don't require scaling
    val_preds = dt.predict(X_val)
    val_r2 = r2_score(y_val, val_preds)
    
    if val_r2 > best_val_r2:
        best_val_r2 = val_r2
        best_criterion = criterion

print(f"Best Decision Tree Criterion: {best_criterion} (Validation R² = {best_val_r2:.4f})")

# Final Model Training
# Re-train DT on original training data
final_dt = DecisionTreeRegressor(criterion=best_criterion, random_state=42)
final_dt.fit(X_train, y_train)

# Train SVR on SCALED training data
final_svr = SVR(kernel='linear')
final_svr.fit(X_train_scaled, y_train)

# Evaluation on Test Set
y_pred_dt = final_dt.predict(X_test)
test_r2_dt = r2_score(y_test, y_pred_dt)

y_pred_svr = final_svr.predict(X_test_scaled)
test_r2_svr = r2_score(y_test, y_pred_svr)

print("\n--- Final R-squared Scores (Test Set) ---")
print(f"Decision Tree Regressor R² Score: {test_r2_dt:.4f}")
print(f"Support Vector Regressor R² Score: {test_r2_svr:.4f}")

Missing values per column before cleaning:
Product_Code            0
Warehouse               0
Product_Category        0
Date                11239
Order_Demand            0
dtype: int64

Missing values after cleaning:
Product_Code        0
Warehouse           0
Product_Category    0
Date                0
Order_Demand        0
dtype: int64
Processed feature shape: (1037336, 37)
Train set size:      829868 rows (80%)
Validation set size: 103734 rows (10%)
Test set size:        103734 rows (10%)
Best Decision Tree Criterion: friedman_mse (Validation R² = 0.0896)

--- Final R-squared Scores (Test Set) ---
Decision Tree Regressor R² Score: 0.1140
Support Vector Regressor R² Score: 0.4897
